## Bronze

carrega os csvs da landing sem tratamento
se um arquivo for carregado novamente, substitui com o replaceWhere

In [0]:
from pyspark.sql import functions as F

# define onde chegam os csv
LANDING = "/Volumes/novarota/bronze/landing"
BRONZE = "novarota.bronze"

# uma tabela bronze por arquivo
TABELAS = ["clientes", "contas", "cartoes", "estornos", "transacoes"]

In [0]:
# pastas que existem na landing (cada pasta é uma data de carga)
datas_landing = sorted(pasta.name.replace("/", "") for pasta in dbutils.fs.ls(LANDING))

# datas que já estão na bronze. transacoes como referencia pois tem a data no nome
if spark.catalog.tableExists(f"{BRONZE}.transacoes"):
    linhas = spark.sql(f"select distinct data_referencia from {BRONZE}.transacoes").collect()
    carregadas = [linha.data_referencia for linha in linhas]
else:
    carregadas = []

datas_novas = [d for d in datas_landing if d not in carregadas]

print("na landing:", datas_landing)
print("já carregadas:", carregadas)
print("pra carregar agora:", datas_novas)

In [0]:
for data_ref in datas_novas:
    print(f"carregando {data_ref}")

    for tabela in TABELAS:
        # usando transacoes pois tem a data no nome
        if tabela == "transacoes":
            arquivo = f"transacoes_cartao_{data_ref}.csv"
        else:
            arquivo = f"{tabela}.csv"

        # carrega tudo como texto pra nao correr risco de quebrar
        df = spark.read.option("header", True).csv(f"{LANDING}/{data_ref}/{arquivo}")

        # trocar nulo por vazio antes de fazer o concat pq só o concat pula nulo e pode quebrar o hash
        colunas_sem_nulo = [F.coalesce(F.col(c), F.lit("")) for c in df.columns]

        # colunas de controle
        df = (
            df.withColumn("arquivo_origem", F.col("_metadata.file_path"))
              .withColumn("data_referencia", F.lit(data_ref))
              .withColumn("timestamp_ingestao", F.current_timestamp())
              .withColumn("hash_linha", F.sha2(F.concat_ws("|", *colunas_sem_nulo), 256))
        )

        if spark.catalog.tableExists(f"{BRONZE}.{tabela}"):
            # se a tabela ja existe, mexe só nas linhas dessa data
            (df.write.format("delta")
               .mode("overwrite")
               .option("replaceWhere", f"data_referencia = '{data_ref}'")
               .option("mergeSchema", "true")  # se vier coluna nova no arquivo, ela entra na tabela
               .saveAsTable(f"{BRONZE}.{tabela}"))
        else:
            # primeira carga cria a tabela
            df.write.format("delta").saveAsTable(f"{BRONZE}.{tabela}")

        print(f"{BRONZE}.{tabela} | {data_ref} | {df.count()} linhas")

if not datas_novas:
    print("nada novo na landing")

In [0]:
%sql
-- quantas linhas entraram em cada tabela da bronze
select 'clientes' as tabela, data_referencia, count(*) as linhas from novarota.bronze.clientes group by data_referencia
union all
select 'contas', data_referencia, count(*) from novarota.bronze.contas group by data_referencia
union all
select 'cartoes', data_referencia, count(*) from novarota.bronze.cartoes group by data_referencia
union all
select 'estornos', data_referencia, count(*) from novarota.bronze.estornos group by data_referencia
union all
select 'transacoes', data_referencia, count(*) from novarota.bronze.transacoes group by data_referencia
order by tabela, data_referencia